Step 1 — Load Silver table

In [0]:
from pyspark.sql.functions import *

silver_path = "/Volumes/smart_fraud_databricks/default/raw_data/silver"

accounts_silver = spark.read.format("delta").load(
    f"{silver_path}/accounts"
)

transactions_silver = spark.read.format("delta").load(
    f"{silver_path}/transactions"
)

watchlist_silver = spark.read.format("delta").load(
    f"{silver_path}/fraud_watchlist"
)

print("Silver data loaded successfully")

Silver data loaded successfully


Step 2 — Check the columns

In [0]:
print("ACCOUNT COLUMNS:")
print(accounts_silver.columns)

print("\nTRANSACTION COLUMNS:")
print(transactions_silver.columns)

print("\nWATCHLIST COLUMNS:")
print(watchlist_silver.columns)

ACCOUNT COLUMNS:
['account_id', 'customer_name', 'account_type', 'credit_limit', 'branch']

TRANSACTION COLUMNS:
['txn_id', 'account_id', 'txn_date', 'amount', 'merchant']

WATCHLIST COLUMNS:
['account_id', 'fraud_type', 'flagged_date']


Step 3 — Display sample records

In [0]:
display(accounts_silver.limit(5))

account_id,customer_name,account_type,credit_limit,branch
ACC0295,Customer_295,Current,500000,Bangalore
ACC0318,Customer_318,Current,500000,Delhi
ACC0238,Customer_238,Current,200000,Pune
ACC0136,Customer_136,Savings,50000,Chennai
ACC0436,Customer_436,Current,100000,Chennai


In [0]:
display(transactions_silver.limit(5))

txn_id,account_id,txn_date,amount,merchant
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket
TXN000878,ACC0466,2025-03-30,72273.67,Zomato
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket


In [0]:
display(watchlist_silver.limit(5))

account_id,fraud_type,flagged_date
ACC0220,Card Skimming,2025-05-04
ACC0192,Phishing,2025-05-07
ACC0050,Card Skimming,2025-02-12
ACC0280,Account Takeover,2025-03-03
ACC0178,Card Skimming,2025-02-17


Step 4 — Join Transactions + Accounts

In [0]:
gold_df = transactions_silver.alias("t") \
    .join(
        accounts_silver.alias("a"),
        col("t.account_id") == col("a.account_id"),
        "left"
    ) \
    .select(
        col("t.txn_id"),
        col("t.account_id"),
        col("t.txn_date"),
        col("t.amount"),
        col("t.merchant"),
        col("a.customer_name"),
        col("a.account_type"),
        col("a.credit_limit"),
        col("a.branch")
    )

display(gold_df.limit(10))

txn_id,account_id,txn_date,amount,merchant,customer_name,account_type,credit_limit,branch
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal,Customer_410,Savings,200000,Delhi
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket,Customer_476,Savings,500000,Pune
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket,Customer_159,Salary,200000,Chennai
TXN000878,ACC0466,2025-03-30,72273.67,Zomato,Customer_466,Current,100000,Pune
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket,Customer_438,Savings,500000,Mumbai
TXN015224,ACC0233,2025-05-24,109450.52,Unknown_POS,Customer_233,Current,100000,Chennai
TXN014385,ACC0455,2025-07-05,49937.67,Flipkart,Customer_455,Salary,500000,Mumbai
TXN007293,ACC0452,2025-07-12,100834.15,Flipkart,Customer_452,Savings,200000,Bangalore
TXN013059,ACC0349,2025-05-27,56120.37,Unknown_POS,Customer_349,Salary,500000,Pune
TXN004388,ACC0187,2025-07-15,387.61,Swiggy,Customer_187,Savings,100000,Chennai


Step 5 — Add Watchlist Information

In [0]:
gold_df = gold_df.alias("g") \
    .join(
        watchlist_silver.alias("w"),
        col("g.account_id") == col("w.account_id"),
        "left"
    ) \
    .select(
        col("g.*"),
        col("w.fraud_type"),
        col("w.flagged_date")
    )

In [0]:
display(gold_df.limit(10))

txn_id,account_id,txn_date,amount,merchant,customer_name,account_type,credit_limit,branch,fraud_type,flagged_date
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal,Customer_410,Savings,200000,Delhi,null,null
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket,Customer_476,Savings,500000,Pune,null,null
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket,Customer_159,Salary,200000,Chennai,null,null
TXN000878,ACC0466,2025-03-30,72273.67,Zomato,Customer_466,Current,100000,Pune,null,null
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket,Customer_438,Savings,500000,Mumbai,null,null
TXN015224,ACC0233,2025-05-24,109450.52,Unknown_POS,Customer_233,Current,100000,Chennai,Account Takeover,2025-05-09
TXN014385,ACC0455,2025-07-05,49937.67,Flipkart,Customer_455,Salary,500000,Mumbai,null,null
TXN007293,ACC0452,2025-07-12,100834.15,Flipkart,Customer_452,Savings,200000,Bangalore,null,null
TXN013059,ACC0349,2025-05-27,56120.37,Unknown_POS,Customer_349,Salary,500000,Pune,Card Skimming,2025-05-25
TXN004388,ACC0187,2025-07-15,387.61,Swiggy,Customer_187,Savings,100000,Chennai,null,null


Step 6 — Create Fraud Indicator

In [0]:
gold_df = gold_df.withColumn(
    "is_watchlisted",
    when(col("fraud_type").isNotNull(), 1).otherwise(0)
)

In [0]:
display(
    gold_df.groupBy("is_watchlisted").count()
)

is_watchlisted,count
1,1685
0,18366


Step 7 — Create Amount-Based Fraud Features

In [0]:
gold_df = gold_df.withColumn(
    "amount_to_credit_ratio",
    when(
        col("credit_limit") > 0,
        col("amount") / col("credit_limit")
    ).otherwise(0)
)

In [0]:
gold_df = gold_df.withColumn(
    "is_high_value",
    when(col("amount_to_credit_ratio") >= 0.8, 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import col, when

# Convert numeric columns safely
gold_df = gold_df.withColumn(
    "amount",
    col("amount").cast("double")
)

gold_df = gold_df.withColumn(
    "credit_limit",
    col("credit_limit").cast("double")
)

# Calculate transaction amount / credit limit
gold_df = gold_df.withColumn(
    "amount_to_credit_ratio",
    when(
        col("credit_limit").isNotNull() & (col("credit_limit") > 0),
        col("amount") / col("credit_limit")
    ).otherwise(0.0)
)

# High-value transaction indicator
gold_df = gold_df.withColumn(
    "is_high_value",
    when(col("amount_to_credit_ratio") >= 0.8, 1)
    .otherwise(0)
)

display(gold_df.limit(10))

txn_id,account_id,txn_date,amount,merchant,customer_name,account_type,credit_limit,branch,fraud_type,flagged_date,is_watchlisted,amount_to_credit_ratio,is_high_value
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal,Customer_410,Savings,200000.0,Delhi,null,null,0,0.64683985,0
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket,Customer_476,Savings,500000.0,Pune,null,null,0,0.05303524,0
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket,Customer_159,Salary,200000.0,Chennai,null,null,0,0.4990869,0
TXN000878,ACC0466,2025-03-30,72273.67,Zomato,Customer_466,Current,100000.0,Pune,null,null,0,0.7227367,0
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket,Customer_438,Savings,500000.0,Mumbai,null,null,0,0.05216968,0
TXN015224,ACC0233,2025-05-24,109450.52,Unknown_POS,Customer_233,Current,100000.0,Chennai,Account Takeover,2025-05-09,1,1.0945052,1
TXN014385,ACC0455,2025-07-05,49937.67,Flipkart,Customer_455,Salary,500000.0,Mumbai,null,null,0,0.09987533999999999,0
TXN007293,ACC0452,2025-07-12,100834.15,Flipkart,Customer_452,Savings,200000.0,Bangalore,null,null,0,0.50417075,0
TXN013059,ACC0349,2025-05-27,56120.37,Unknown_POS,Customer_349,Salary,500000.0,Pune,Card Skimming,2025-05-25,1,0.11224074,0
TXN004388,ACC0187,2025-07-15,387.61,Swiggy,Customer_187,Savings,100000.0,Chennai,null,null,0,0.0038761,0


In [0]:
gold_df.select(
    "amount",
    "credit_limit",
    "amount_to_credit_ratio",
    "is_high_value"
).printSchema()

root
 |-- amount: double (nullable = true)
 |-- credit_limit: double (nullable = true)
 |-- amount_to_credit_ratio: double (nullable = true)
 |-- is_high_value: integer (nullable = false)



Step 8 — Create Final Gold Dataset

In [0]:
gold_final = gold_df.select(
    "txn_id",
    "account_id",
    "txn_date",
    "amount",
    "merchant",
    "customer_name",
    "account_type",
    "credit_limit",
    "branch",
    "fraud_type",
    "flagged_date",
    "is_watchlisted",
    "amount_to_credit_ratio",
    "is_high_value"
)

display(gold_final.limit(20))

txn_id,account_id,txn_date,amount,merchant,customer_name,account_type,credit_limit,branch,fraud_type,flagged_date,is_watchlisted,amount_to_credit_ratio,is_high_value
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal,Customer_410,Savings,200000.0,Delhi,null,null,0,0.64683985,0
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket,Customer_476,Savings,500000.0,Pune,null,null,0,0.05303524,0
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket,Customer_159,Salary,200000.0,Chennai,null,null,0,0.4990869,0
TXN000878,ACC0466,2025-03-30,72273.67,Zomato,Customer_466,Current,100000.0,Pune,null,null,0,0.7227367,0
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket,Customer_438,Savings,500000.0,Mumbai,null,null,0,0.05216968,0
TXN015224,ACC0233,2025-05-24,109450.52,Unknown_POS,Customer_233,Current,100000.0,Chennai,Account Takeover,2025-05-09,1,1.0945052,1
TXN014385,ACC0455,2025-07-05,49937.67,Flipkart,Customer_455,Salary,500000.0,Mumbai,null,null,0,0.09987533999999999,0
TXN007293,ACC0452,2025-07-12,100834.15,Flipkart,Customer_452,Savings,200000.0,Bangalore,null,null,0,0.50417075,0
TXN013059,ACC0349,2025-05-27,56120.37,Unknown_POS,Customer_349,Salary,500000.0,Pune,Card Skimming,2025-05-25,1,0.11224074,0
TXN004388,ACC0187,2025-07-15,387.61,Swiggy,Customer_187,Savings,100000.0,Chennai,null,null,0,0.0038761,0


Step 11 — Save Gold

In [0]:
gold_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold"

gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("Gold Delta layer created successfully!")

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8977817292198545>, line 6
      1 gold_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold"
      3 gold_final.write \
      4     .format("delta") \
      5     .mode("overwrite") \
----> 6     .save(gold_path)
      8 print("Gold Delta layer created successfully!")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observa

In [0]:
from pyspark.sql.functions import col, expr

silver_path = "/Volumes/smart_fraud_databricks/default/raw_data/silver"

accounts_silver = spark.read.format("delta").load(
    f"{silver_path}/accounts"
)

transactions_silver = spark.read.format("delta").load(
    f"{silver_path}/transactions"
)

watchlist_silver = spark.read.format("delta").load(
    f"{silver_path}/fraud_watchlist"
)

In [0]:
accounts_silver.select("account_id", "credit_limit").show(20, False)

+----------+------------+
|account_id|credit_limit|
+----------+------------+
|ACC0295   |500000      |
|ACC0318   |500000      |
|ACC0238   |200000      |
|ACC0136   |50000       |
|ACC0436   |100000      |
|ACC0369   |50000       |
|ACC0501   |150000      |
|ACC0401   |200000      |
|ACC0354   |200000      |
|ACC0435   |200000      |
|ACC0118   |200000      |
|ACC0035   |100000      |
|ACC0404   |50000       |
|ACC0063   |200000      |
|ACC0111   |200000      |
|ACC0041   |500000      |
|ACC0487   |500000      |
|ACC0476   |500000      |
|ACC0412   |200000      |
|ACC0028   |100000      |
+----------+------------+
only showing top 20 rows


In [0]:
accounts_silver = accounts_silver.withColumn(
    "credit_limit",
    expr("try_cast(cast(credit_limit AS STRING) AS DOUBLE)")
)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "amount",
    expr("try_cast(cast(amount AS STRING) AS DOUBLE)")
)

In [0]:
gold_df = transactions_silver.alias("t") \
    .join(
        accounts_silver.alias("a"),
        col("t.account_id") == col("a.account_id"),
        "left"
    ) \
    .select(
        col("t.txn_id"),
        col("t.account_id"),
        col("t.txn_date"),
        col("t.amount"),
        col("t.merchant"),
        col("a.customer_name"),
        col("a.account_type"),
        col("a.credit_limit"),
        col("a.branch")
    )

In [0]:
gold_df = gold_df.alias("g") \
    .join(
        watchlist_silver.alias("w"),
        col("g.account_id") == col("w.account_id"),
        "left"
    ) \
    .select(
        col("g.*"),
        col("w.fraud_type"),
        col("w.flagged_date")
    )

In [0]:
gold_df = gold_df.withColumn(
    "is_watchlisted",
    when(col("fraud_type").isNotNull(), 1).otherwise(0)
)

gold_df = gold_df.withColumn(
    "amount_to_credit_ratio",
    when(
        (col("credit_limit") > 0) & col("amount").isNotNull(),
        col("amount") / col("credit_limit")
    ).otherwise(0.0)
)

gold_df = gold_df.withColumn(
    "is_high_value",
    when(col("amount_to_credit_ratio") >= 0.8, 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import *

In [0]:
gold_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("✅ GOLD DELTA CREATED SUCCESSFULLY")

✅ GOLD DELTA CREATED SUCCESSFULLY
